# ⚡ StudyBook AI Engine on Kaggle 2x NVIDIA T4 GPUs

This notebook hosts the backend for **StudyBook** using Kaggle's dual **2x NVIDIA T4 (32 GB VRAM)**:
- **2x T4 GPUs (32 GB VRAM)**: Qwen2.5-7B-Instruct in FP16 distributed across both GPUs () for 70-100 tok/sec throughput
- **Security**: API Key authentication required for every request
- **Diagnostics**:  and  endpoints for live connection verification
- **CPU Engine**: BAAI/bge-small embeddings, multi-threaded Edge-TTS podcast synthesizer, and Faster-Whisper
- **Instant Tunnel**: Cloudflare Tunnel (100% free, unlimited, no ngrok token needed)


In [ ]:
# Step 1: Install Dependencies & Cloudflare Tunnel
!pip install -q fastapi uvicorn python-multipart pydantic httpx transformers accelerate sentence-transformers edge-tts faster-whisper bitsandbytes
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Install Cloudflare Tunnel binary
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print("✓ Dependencies and cloudflared installed successfully!")


In [ ]:
# Step 2: Diagnostic Check for 2x T4 GPUs
import torch
num_gpus = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Detected GPUs: {num_gpus}")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.1f} GB")


In [ ]:
# Step 3: Run the All-in-One 2x T4 StudyBook Server with API Key & Health Endpoints
import os, sys, gc
os.environ["KAGGLE_API_KEY"] = "studybook-t4-secret"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Evict previous server module and free any lingering CUDA VRAM
if "kaggle_backend_server" in sys.modules:
    del sys.modules["kaggle_backend_server"]
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

# Write server script
with open("kaggle_backend_server.py", "w") as f:
    f.write("\"\"\"\nKaggle 2x T4 GPU All-in-One AI Backend Server for StudyBook\n===========================================================\nEngineered for Kaggle's Dual NVIDIA Tesla T4 GPUs (2x 16 GB = 32 GB VRAM):\n- Parallel GPU Execution: Qwen2.5-7B-Instruct in FP16 distributed across both T4 GPUs\n  (via tensor parallelism / device_map=\"auto\") for 70-100+ tokens/second throughput.\n- Vector Embeddings: BAAI/bge-small-en-v1.5 (high speed, 384/512-dim normalized vectors).\n- Podcast Audio Engine: Multi-threaded async Edge-TTS + FFmpeg stitching for 10-15 min episodes.\n- STT: Faster-Whisper (base.en).\n- Tunnel: Cloudflare Tunnel (100% free, permanent URL, no tokens or warning screens).\n\nEndpoints Exposed:\n- GET  /health                  -> Health check & GPU diagnostics (GPU 0 and GPU 1 status)\n- GET  /v1/models               -> Discovered model catalog\n- POST /v1/chat/completions     -> OpenAI-compatible chat (Notebook Chat, Flashcards, Deep Ask, Transformations)\n- POST /v1/embeddings           -> OpenAI-compatible text embeddings (Vector RAG)\n- POST /api/podcast             -> 10-15 minute narrative podcast generator (7B script + async audio stitching)\n- POST /v1/audio/transcriptions -> Whisper STT\n\"\"\"\n\nimport os\nos.environ[\"PYTORCH_CUDA_ALLOC_CONF\"] = \"expandable_segments:True\"\n\nimport asyncio\nimport base64\nfrom contextlib import asynccontextmanager\nimport io\nimport json\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom typing import Any, Dict, List, Optional, Union\n\nimport uvicorn\nfrom fastapi import FastAPI, HTTPException, Request, Response, UploadFile, File\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import JSONResponse, StreamingResponse\nfrom pydantic import BaseModel, Field\n\n# ==============================================================================\n# Model Configurations (Optimized for 2x NVIDIA T4 GPUs)\n# ==============================================================================\nLLM_MODEL_ID = \"Qwen/Qwen2.5-7B-Instruct\"  # Blazing fast, 128k context, strict structured JSON\nEMBEDDING_MODEL_ID = \"BAAI/bge-small-en-v1.5\"  # High accuracy, low memory, fast embedding model\nDEFAULT_VOICE_EN = \"en-US-ChristopherNeural\"  # Natural, deep, warm American male host\nDEFAULT_VOICE_HI = \"hi-IN-MadhurNeural\"       # Native warm conversational Hindi male host\n\n# API Authentication Configuration\n# Default key is 'studybook-t4-secret' or set via KAGGLE_API_KEY environment variable.\nAPI_KEY = os.environ.get(\"KAGGLE_API_KEY\", os.environ.get(\"STUDYBOOK_API_KEY\", \"studybook-t4-secret\"))\n\n# Global model state\nllm_pipeline = None\nembedder = None\nwhisper_model = None\n\n\n@asynccontextmanager\nasync def lifespan(app: FastAPI):\n    init_embedding_engine()\n    init_whisper_engine()\n    loop = asyncio.get_running_loop()\n    loop.run_in_executor(None, init_llm_engine)\n    yield\n\n\n# ==============================================================================\n# FastAPI App Initialization\n# ==============================================================================\napp = FastAPI(title=\"StudyBook Kaggle 2x T4 GPU Backend\", version=\"1.0.0\", lifespan=lifespan)\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=[\"*\"],\n    allow_credentials=True,\n    allow_methods=[\"*\"],\n    allow_headers=[\"*\"],\n)\n\n\n@app.middleware(\"http\")\nasync def require_api_key(request: Request, call_next):\n    # Allow CORS preflight requests\n    if request.method == \"OPTIONS\":\n        return await call_next(request)\n\n    # Check Authorization header (Bearer <key>), x-api-key header, or query param (?api_key= or ?token=)\n    auth_header = request.headers.get(\"Authorization\") or \"\"\n    x_api_key = request.headers.get(\"x-api-key\") or \"\"\n    query_key = request.query_params.get(\"api_key\") or request.query_params.get(\"token\") or \"\"\n\n    token = \"\"\n    if auth_header.startswith(\"Bearer \"):\n        token = auth_header[7:].strip()\n    elif auth_header:\n        token = auth_header.strip()\n    elif x_api_key:\n        token = x_api_key.strip()\n    elif query_key:\n        token = query_key.strip()\n\n    if not token or token != API_KEY:\n        return JSONResponse(\n            status_code=401,\n            content={\n                \"error\": \"Unauthorized\",\n                \"detail\": \"Invalid or missing API key. Include 'Authorization: Bearer <key>', 'x-api-key: <key>', or '?api_key=<key>'.\",\n            },\n        )\n\n    return await call_next(request)\n\n\n# ==============================================================================\n# Model Loaders\n# ==============================================================================\ndef init_embedding_engine():\n    global embedder\n    print(\"[1/3] Loading Embedding Engine (BAAI/bge-small-en-v1.5)...\")\n    try:\n        from sentence_transformers import SentenceTransformer\n        # Embeddings run on CPU to leave 100% of GPU memory for the 7B LLM\n        embedder = SentenceTransformer(EMBEDDING_MODEL_ID, device=\"cpu\")\n        print(\"\u2713 Embedding Engine loaded on CPU.\")\n    except Exception as e:\n        print(f\"Embedding Engine warning: {e}. Falling back to FastEmbed/dummy.\")\n\n\ndef init_llm_engine():\n    global llm_pipeline\n    print(f\"[2/3] Initializing Large Language Model: {LLM_MODEL_ID} across 2x T4 GPUs...\")\n    try:\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline\n\n        num_gpus = torch.cuda.device_count()\n        print(f\"Detected CUDA devices: {num_gpus}\")\n        for i in range(num_gpus):\n            props = torch.cuda.get_device_properties(i)\n            print(f\"  GPU {i}: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)\")\n\n        tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)\n\n        model = None\n        # Option A: Fast 4-bit NF4 quantization (Uses only ~4.5 GB total VRAM, leaving >10 GB free on each GPU)\n        try:\n            from transformers import BitsAndBytesConfig\n            bnb_config = BitsAndBytesConfig(\n                load_in_4bit=True,\n                bnb_4bit_quant_type=\"nf4\",\n                bnb_4bit_compute_dtype=torch.float16,\n                bnb_4bit_use_double_quant=True,\n            )\n            print(\"Attempting 4-bit NF4 quantization (blazing speed & >10 GB free VRAM buffer)...\")\n            model = AutoModelForCausalLM.from_pretrained(\n                LLM_MODEL_ID,\n                quantization_config=bnb_config,\n                device_map=\"auto\",\n                trust_remote_code=True,\n            )\n            print(\"\u2713 Qwen2.5-7B-Instruct loaded in 4-bit NF4 successfully! (~4.5 GB total VRAM)\")\n        except Exception as bnb_err:\n            print(f\"4-bit quantization fallback ({bnb_err}), using explicit 50/50 dual-GPU layer distribution...\")\n\n        # Option B: Explicit 50/50 layer split in FP16 (7.6 GB on GPU 0, 7.6 GB on GPU 1)\n        if model is None:\n            # True 50/50 balance: embed_tokens (1.1 GB) + layers 0-10 (5.7 GB) = 6.8 GB on GPU 0\n            # Layers 11-27 (8.8 GB) + norm/lm_head (0.1 GB) = 8.9 GB on GPU 1\n            # Leaves ~7.8 GB of FREE VRAM on GPU 0 for attention & KV cache!\n            layer_map = {\"model.embed_tokens\": 0}\n            for i in range(28):\n                layer_map[f\"model.layers.{i}\"] = 0 if i <= 10 else 1\n            layer_map[\"model.norm\"] = 1\n            layer_map[\"lm_head\"] = 1\n\n            print(\"Assigning balanced layers (embed + layers 0-10 -> GPU 0, layers 11-27 -> GPU 1)...\")\n            model = AutoModelForCausalLM.from_pretrained(\n                LLM_MODEL_ID,\n                torch_dtype=torch.float16,\n                device_map=layer_map,\n                attn_implementation=\"sdpa\",\n                trust_remote_code=True,\n                low_cpu_mem_usage=True,\n            )\n            print(\"\u2713 Qwen2.5-7B-Instruct evenly split across GPU 0 (~6.8 GB) and GPU 1 (~8.9 GB) with SDPA attention!\")\n\n        model.eval()\n\n        # Print layer distribution per device to verify both GPUs are utilized\n        if hasattr(model, \"hf_device_map\"):\n            devices_used = set(model.hf_device_map.values())\n            print(f\"\u2713 Layers mapped to devices: {devices_used}\")\n\n        llm_pipeline = pipeline(\n            \"text-generation\",\n            model=model,\n            tokenizer=tokenizer,\n            torch_dtype=torch.float16,\n        )\n        print(\"\u2713 LLM Pipeline initialized successfully!\")\n    except Exception as e:\n        print(f\"LLM initialization warning: {e}\")\n        try:\n            from transformers import pipeline\n            fallback_id = \"Qwen/Qwen2.5-1.5B-Instruct\"\n            print(f\"Falling back to lightweight model {fallback_id}...\")\n            llm_pipeline = pipeline(\"text-generation\", model=fallback_id, device_map=\"auto\")\n            print(\"\u2713 Fallback LLM initialized.\")\n        except Exception as ex:\n            print(f\"Fatal LLM error: {ex}\")\n\n\ndef init_whisper_engine():\n    global whisper_model\n    print(\"[3/3] Loading Whisper Speech-to-Text Engine...\")\n    try:\n        from faster_whisper import WhisperModel\n        whisper_model = WhisperModel(\"base.en\", device=\"cpu\", compute_type=\"int8\", cpu_threads=4)\n        print(\"\u2713 Whisper Engine loaded on CPU.\")\n    except Exception as e:\n        print(f\"Whisper STT note: {e}\")\n\n\n# ==============================================================================\n# Endpoint 1: Root, Ping, and Health Diagnostics\n# ==============================================================================\n@app.get(\"/\")\nasync def root():\n    return {\n        \"status\": \"online\",\n        \"backend\": \"Kaggle 2x T4 GPU\",\n        \"message\": \"StudyBook Kaggle Backend is healthy and ready.\",\n        \"endpoints\": [\"/ping\", \"/health\", \"/v1/models\", \"/v1/chat/completions\", \"/v1/embeddings\", \"/api/podcast\"],\n    }\n\n\n@app.get(\"/ping\")\n@app.head(\"/ping\")\nasync def ping():\n    return {\n        \"status\": \"ok\",\n        \"message\": \"pong\",\n        \"backend\": \"Kaggle 2x T4 GPU\",\n    }\n\n\n@app.get(\"/health\")\n@app.head(\"/health\")\nasync def health_check():\n    import torch\n    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0\n    gpu_info = []\n    for i in range(num_gpus):\n        props = torch.cuda.get_device_properties(i)\n        gpu_info.append({\n            \"index\": i,\n            \"name\": props.name,\n            \"vram_gb\": round(props.total_memory / (1024**3), 1),\n            \"allocated_gb\": round(torch.cuda.memory_allocated(i) / (1024**3), 2),\n        })\n\n    return {\n        \"status\": \"healthy\",\n        \"backend\": \"Kaggle 2x T4 GPU\",\n        \"gpu_count\": num_gpus,\n        \"gpus\": gpu_info,\n        \"llm_ready\": llm_pipeline is not None,\n        \"embedding_ready\": embedder is not None,\n        \"stt_ready\": whisper_model is not None,\n        \"models\": {\n            \"llm\": LLM_MODEL_ID,\n            \"embedding\": EMBEDDING_MODEL_ID,\n            \"tts\": \"Edge-TTS Neural Multi-threaded\",\n        },\n    }\n\n\n# ==============================================================================\n# Endpoint 2: Model Discovery\n# ==============================================================================\n@app.get(\"/v1/models\")\nasync def list_models():\n    return {\n        \"object\": \"list\",\n        \"data\": [\n            {\n                \"id\": \"Qwen/Qwen2.5-7B-Instruct\",\n                \"object\": \"model\",\n                \"type\": \"language\",\n                \"owned_by\": \"kaggle\",\n                \"description\": f\"Dual-T4 accelerated LLM ({LLM_MODEL_ID})\",\n            },\n            {\n                \"id\": \"kaggle-tpu-chat\",\n                \"object\": \"model\",\n                \"type\": \"language\",\n                \"owned_by\": \"kaggle\",\n                \"description\": f\"Dual-T4 accelerated LLM ({LLM_MODEL_ID})\",\n            },\n            {\n                \"id\": \"BAAI/bge-small-en-v1.5\",\n                \"object\": \"model\",\n                \"type\": \"embedding\",\n                \"owned_by\": \"kaggle\",\n                \"description\": f\"Vector Embeddings ({EMBEDDING_MODEL_ID})\",\n            },\n            {\n                \"id\": \"kaggle-tpu-embedding\",\n                \"object\": \"model\",\n                \"type\": \"embedding\",\n                \"owned_by\": \"kaggle\",\n                \"description\": f\"Vector Embeddings ({EMBEDDING_MODEL_ID})\",\n            },\n            {\n                \"id\": \"kaggle-tpu-podcast\",\n                \"object\": \"model\",\n                \"type\": \"text_to_speech\",\n                \"owned_by\": \"kaggle\",\n                \"description\": \"Multi-threaded Neural Podcast Engine\",\n            },\n            {\n                \"id\": \"kaggle-tpu-stt\",\n                \"object\": \"model\",\n                \"type\": \"speech_to_text\",\n                \"owned_by\": \"kaggle\",\n                \"description\": \"Whisper Speech-to-Text\",\n            },\n        ],\n    }\n\n\n# ==============================================================================\n# Endpoint 3: OpenAI-Compatible Chat Completions\n# ==============================================================================\nclass ChatMessage(BaseModel):\n    role: str\n    content: str\n\n\nclass ChatCompletionRequest(BaseModel):\n    model: str = \"kaggle-tpu-chat\"\n    messages: List[ChatMessage]\n    temperature: Optional[float] = 0.2\n    max_tokens: Optional[int] = 8192\n    stream: Optional[bool] = False\n\n\ndef _format_messages_to_prompt(messages: List[ChatMessage], tokenizer) -> str:\n    if hasattr(tokenizer, \"apply_chat_template\"):\n        hf_messages = [{\"role\": m.role, \"content\": m.content} for m in messages]\n        try:\n            return tokenizer.apply_chat_template(hf_messages, tokenize=False, add_generation_prompt=True)\n        except Exception:\n            pass\n\n    blocks = []\n    for m in messages:\n        if m.role == \"system\":\n            blocks.append(f\"<|im_start|>system\\n{m.content}<|im_end|>\")\n        elif m.role == \"user\":\n            blocks.append(f\"<|im_start|>user\\n{m.content}<|im_end|>\")\n        elif m.role == \"assistant\":\n            blocks.append(f\"<|im_start|>assistant\\n{m.content}<|im_end|>\")\n    blocks.append(\"<|im_start|>assistant\\n\")\n    return \"\\n\".join(blocks)\n\n\n@app.post(\"/v1/chat/completions\")\nasync def chat_completions(req: ChatCompletionRequest):\n    if llm_pipeline is None:\n        raise HTTPException(status_code=503, detail=\"LLM engine is still initializing on 2x T4. Please retry in 30 seconds.\")\n\n    prompt_text = _format_messages_to_prompt(req.messages, llm_pipeline.tokenizer)\n    max_tokens = min(req.max_tokens or 8192, 16384)\n    temp = max(0.01, req.temperature or 0.2)\n\n    def _generate():\n        import torch\n        with torch.inference_mode():\n            outputs = llm_pipeline(\n                prompt_text,\n                max_new_tokens=max_tokens,\n                temperature=temp,\n                do_sample=temp > 0.05,\n                pad_token_id=getattr(llm_pipeline.tokenizer, \"eos_token_id\", None),\n                return_full_text=False,\n            )\n            res = outputs[0][\"generated_text\"]\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        return res\n\n    loop = asyncio.get_event_loop()\n    generated_text = await loop.run_in_executor(None, _generate)\n\n    # Clean thinking tags if present\n    cleaned_text = re.sub(r\"<think>[\\s\\S]*?</think>\", \"\", generated_text).strip()\n    if not cleaned_text:\n        cleaned_text = generated_text.strip()\n\n    # Streaming mode response\n    if req.stream:\n        async def event_generator():\n            chunk_size = 12\n            for i in range(0, len(cleaned_text), chunk_size):\n                chunk = cleaned_text[i:i + chunk_size]\n                payload = {\n                    \"id\": f\"chatcmpl-{int(time.time())}\",\n                    \"object\": \"chat.completion.chunk\",\n                    \"created\": int(time.time()),\n                    \"model\": req.model,\n                    \"choices\": [\n                        {\"index\": 0, \"delta\": {\"content\": chunk}, \"finish_reason\": None}\n                    ],\n                }\n                yield f\"data: {json.dumps(payload)}\\n\\n\"\n                await asyncio.sleep(0.015)\n            # Final chunk\n            final_payload = {\n                \"id\": f\"chatcmpl-{int(time.time())}\",\n                \"object\": \"chat.completion.chunk\",\n                \"created\": int(time.time()),\n                \"model\": req.model,\n                \"choices\": [{\"index\": 0, \"delta\": {}, \"finish_reason\": \"stop\"}],\n            }\n            yield f\"data: {json.dumps(final_payload)}\\n\\n\"\n            yield \"data: [DONE]\\n\\n\"\n\n        return StreamingResponse(event_generator(), media_type=\"text/event-stream\")\n\n    # Non-streaming response\n    return {\n        \"id\": f\"chatcmpl-{int(time.time())}\",\n        \"object\": \"chat.completion\",\n        \"created\": int(time.time()),\n        \"model\": req.model,\n        \"choices\": [\n            {\n                \"index\": 0,\n                \"message\": {\n                    \"role\": \"assistant\",\n                    \"content\": cleaned_text,\n                },\n                \"finish_reason\": \"stop\",\n            }\n        ],\n        \"usage\": {\n            \"prompt_tokens\": len(prompt_text) // 4,\n            \"completion_tokens\": len(cleaned_text) // 4,\n            \"total_tokens\": (len(prompt_text) + len(cleaned_text)) // 4,\n        },\n    }\n\n\n# ==============================================================================\n# Endpoint 4: Vector Embeddings\n# ==============================================================================\nclass EmbeddingRequest(BaseModel):\n    model: str = \"kaggle-tpu-embedding\"\n    input: Union[str, List[str]]\n\n\n@app.post(\"/v1/embeddings\")\nasync def create_embeddings(req: EmbeddingRequest):\n    if embedder is None:\n        raise HTTPException(status_code=503, detail=\"Embedding engine is still loading.\")\n\n    texts = [req.input] if isinstance(req.input, str) else req.input\n    if not texts:\n        return {\"object\": \"list\", \"data\": [], \"model\": req.model}\n\n    loop = asyncio.get_event_loop()\n    vectors = await loop.run_in_executor(None, lambda: embedder.encode(texts, normalize_embeddings=True))\n\n    data_items = [\n        {\n            \"object\": \"embedding\",\n            \"index\": idx,\n            \"embedding\": vec.tolist() if hasattr(vec, \"tolist\") else list(vec),\n        }\n        for idx, vec in enumerate(vectors)\n    ]\n\n    return {\n        \"object\": \"list\",\n        \"data\": data_items,\n        \"model\": req.model,\n        \"usage\": {\"total_tokens\": sum(len(t) for t in texts) // 4},\n    }\n\n\n# ==============================================================================\n# Endpoint 5: 10-15 Minute Audio Podcast Engine\n# ==============================================================================\nclass PodcastRequest(BaseModel):\n    content: str\n    briefing: str = \"Produce an articulate, conversational podcast episode explaining the concepts.\"\n    episode_name: str = \"Episode\"\n    duration_minutes: int = 15\n    language: str = \"en\"\n\n\nasync def synthesize_speech_edge(text: str, voice: str, output_path: str):\n    \"\"\"Synthesize text chunk to MP3 using edge-tts (streams directly over network, zero CPU/GPU overhead).\"\"\"\n    import edge_tts\n    communicate = edge_tts.Communicate(text, voice)\n    await communicate.save(output_path)\n\n\ndef split_text_into_speech_chunks(text: str, max_chars: int = 280) -> List[str]:\n    \"\"\"Split text cleanly at sentence boundaries so audio flows naturally.\"\"\"\n    sentences = re.split(r\"(?<=[.!?])\\s+\", text)\n    chunks = []\n    current_chunk = \"\"\n\n    for s in sentences:\n        s = s.strip()\n        if not s:\n            continue\n        if len(current_chunk) + len(s) + 1 <= max_chars:\n            current_chunk = f\"{current_chunk} {s}\".strip()\n        else:\n            if current_chunk:\n                chunks.append(current_chunk)\n            current_chunk = s\n\n    if current_chunk:\n        chunks.append(current_chunk)\n    return chunks\n\n\ndef stitch_mp3_files(chunk_files: List[str], output_file: str):\n    \"\"\"Concatenate MP3 files using ffmpeg.\"\"\"\n    with tempfile.NamedTemporaryFile(\"w\", suffix=\".txt\", delete=False) as concat_list:\n        list_path = concat_list.name\n        for cf in chunk_files:\n            concat_list.write(f\"file '{cf}'\\n\")\n\n    try:\n        subprocess.run(\n            [\n                \"ffmpeg\", \"-y\", \"-f\", \"concat\", \"-safe\", \"0\",\n                \"-i\", list_path, \"-c\", \"copy\", output_file\n            ],\n            check=True,\n            stdout=subprocess.DEVNULL,\n            stderr=subprocess.DEVNULL,\n        )\n    finally:\n        if os.path.exists(list_path):\n            os.remove(list_path)\n\n\ndef get_mp3_duration(file_path: str) -> float:\n    \"\"\"Return duration in seconds using ffprobe, mutagen, or bitrate estimation.\"\"\"\n    try:\n        res = subprocess.run(\n            [\n                \"ffprobe\", \"-v\", \"error\", \"-show_entries\",\n                \"format=duration\", \"-of\", \"default=noprint_wrappers=1:nokey=1\",\n                file_path,\n            ],\n            stdout=subprocess.PIPE,\n            stderr=subprocess.PIPE,\n            text=True,\n            check=True,\n        )\n        return max(0.1, float(res.stdout.strip()))\n    except Exception:\n        try:\n            import mutagen.mp3\n            return max(0.1, float(mutagen.mp3.MP3(file_path).info.length))\n        except Exception:\n            # Edge-TTS default 128kbps stereo MP3 is ~16,000 bytes/sec\n            size = os.path.getsize(file_path) if os.path.exists(file_path) else 0\n            return max(1.0, round(size / 16000.0, 2))\n\n\n# Background Podcast Job Registry (Keeps connections under 100ms to eliminate Cloudflare 100s timeout)\nPODCAST_JOBS: Dict[str, Dict[str, Any]] = {}\n\n\nasync def _run_podcast_pipeline(job_id: str, req: PodcastRequest):\n    \"\"\"Executes the full podcast generation pipeline in the background.\"\"\"\n    target_words = 1200 if req.duration_minutes >= 12 else 800\n    target_voice = DEFAULT_VOICE_HI if req.language == \"hi\" else DEFAULT_VOICE_EN\n\n    try:\n        PODCAST_JOBS[job_id][\"progress\"] = 0.15\n        PODCAST_JOBS[job_id][\"message\"] = \"Generating spoken script with Qwen2.5-7B across 2x T4 GPUs...\"\n\n        # Stage 1: Generate spoken monologue on 2x T4\n        script_prompt = f\"\"\"You are a seasoned, charismatic podcast host. Turn the reference material below into an engaging, cohesive {req.duration_minutes}-minute spoken monologue.\n\nREFERENCE MATERIAL:\n{req.content[:10000]}\n\nADDITIONAL BRIEFING:\n{req.briefing}\n\nSTRICT PRODUCTION RULES:\n1. Deliver a natural spoken-word monologue of roughly {target_words} words.\n2. DO NOT include visual markdown (no asterisks, no bullet points, no stage directions like '[Laughs]' or '(Intro music)').\n3. Speak with warmth, eloquence, clarity, and pacing. Introduce the topic with a hook, break down key concepts with vivid analogies, and conclude with insightful synthesis.\n4. Language: {req.language}.\n5. Conclude cleanly with a natural host sign-off sentence. NEVER output hashtags, buzzword lists, metadata tags, emojis, or repeated slogans. Stop speaking immediately after your final sign-off.\n\nBEGIN SPOKEN MONOLOGUE SCRIPT:\"\"\"\n\n        loop = asyncio.get_event_loop()\n        eos_token_id = getattr(getattr(llm_pipeline, \"tokenizer\", None), \"eos_token_id\", None)\n\n        def _generate_script():\n            import torch\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            with torch.inference_mode():\n                outputs = llm_pipeline(\n                    script_prompt,\n                    max_new_tokens=1800,\n                    temperature=0.6,\n                    repetition_penalty=1.18,\n                    no_repeat_ngram_size=4,\n                    do_sample=True,\n                    pad_token_id=eos_token_id,\n                    return_full_text=False,\n                )\n                text = outputs[0][\"generated_text\"]\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            return text\n\n        raw_script = await loop.run_in_executor(None, _generate_script)\n\n        # Remove markdown symbols and stray brackets\n        clean_script = re.sub(r\"[\\*#_\\[\\]\\(\\)\\{\\}\\<\\>]\", \" \", raw_script)\n        # Strip non-standard emojis/pictographs that confuse TTS\n        clean_script = re.sub(r\"[^\\w\\s\\.,!\\?'-]\", \" \", clean_script, flags=re.UNICODE)\n        clean_script = re.sub(r\"\\s+\", \" \", clean_script).strip()\n\n        # Sanitize any trailing degenerate buzzword loop\n        sentences = re.split(r\"(?<=[.!?])\\s+\", clean_script)\n        filtered_sentences = []\n        for s in sentences:\n            words = s.split()\n            # If a sentence is >70% long capitalized words without grammar, it's a buzzword loop\n            title_words = [w for w in words if w.istitle() and len(w) > 3]\n            if len(title_words) >= 8 and len(title_words) / max(1, len(words)) > 0.7:\n                break\n            filtered_sentences.append(s)\n\n        if filtered_sentences:\n            clean_script = \" \".join(filtered_sentences).strip()\n\n        if not clean_script:\n            raise RuntimeError(\"Script generation returned empty text.\")\n\n        PODCAST_JOBS[job_id][\"progress\"] = 0.55\n        PODCAST_JOBS[job_id][\"message\"] = \"Synthesizing parallel Edge-TTS neural speech audio...\"\n\n        # Stage 2: Parallel chunk synthesis (16 concurrent coroutines)\n        chunks = split_text_into_speech_chunks(clean_script)\n        temp_dir = tempfile.mkdtemp()\n\n        try:\n            semaphore = asyncio.Semaphore(16)\n\n            async def _synthesize_chunk(idx: int, chunk_text: str):\n                out_p = os.path.join(temp_dir, f\"chunk_{idx:04d}.mp3\")\n                async with semaphore:\n                    await synthesize_speech_edge(chunk_text, target_voice, out_p)\n                return out_p\n\n            tasks = [_synthesize_chunk(i, c) for i, c in enumerate(chunks)]\n            chunk_paths = await asyncio.gather(*tasks)\n\n            PODCAST_JOBS[job_id][\"progress\"] = 0.88\n            PODCAST_JOBS[job_id][\"message\"] = \"Concatenating MP3 audio chunks into master track...\"\n\n            # Stage 3: Stitch into final MP3\n            master_mp3 = os.path.join(temp_dir, \"master.mp3\")\n            stitch_mp3_files(chunk_paths, master_mp3)\n\n            with open(master_mp3, \"rb\") as f:\n                audio_bytes = f.read()\n\n            audio_b64 = base64.b64encode(audio_bytes).decode(\"utf-8\")\n\n            transcript_data = []\n            curr_time = 0.0\n            for c, c_path in zip(chunks, chunk_paths):\n                dur = get_mp3_duration(c_path)\n                start_sec = round(curr_time, 2)\n                end_sec = round(curr_time + dur, 2)\n                transcript_data.append({\n                    \"speaker\": \"Host\",\n                    \"text\": c,\n                    \"dialogue\": c,\n                    \"start\": start_sec,\n                    \"end\": end_sec,\n                    \"duration\": round(dur, 2),\n                })\n                curr_time += dur\n\n            total_duration = round(curr_time, 2)\n\n            outline_data = {\n                \"title\": req.episode_name,\n                \"duration_target\": f\"{req.duration_minutes} minutes\",\n                \"sections\": [\n                    {\"title\": \"Introduction & Hook\", \"summary\": chunks[0] if chunks else \"\"},\n                    {\"title\": \"Core Breakdown\", \"summary\": \"Detailed exploration of concepts.\"},\n                    {\"title\": \"Synthesis & Conclusion\", \"summary\": chunks[-1] if chunks else \"\"},\n                ],\n            }\n\n            PODCAST_JOBS[job_id] = {\n                \"job_id\": job_id,\n                \"status\": \"completed\",\n                \"progress\": 1.0,\n                \"message\": \"Podcast generated successfully.\",\n                \"result\": {\n                    \"success\": True,\n                    \"episode_name\": req.episode_name,\n                    \"duration_seconds\": total_duration,\n                    \"outline\": outline_data,\n                    \"transcript\": transcript_data,\n                    \"audio_base64\": audio_b64,\n                },\n                \"completed_at\": time.time(),\n            }\n\n        finally:\n            shutil.rmtree(temp_dir, ignore_errors=True)\n\n    except Exception as e:\n        PODCAST_JOBS[job_id] = {\n            \"job_id\": job_id,\n            \"status\": \"failed\",\n            \"progress\": 0.0,\n            \"error\": str(e),\n            \"message\": f\"Failed: {str(e)}\",\n        }\n\n\n@app.post(\"/api/podcast\")\nasync def generate_podcast(req: PodcastRequest, sync: bool = False):\n    \"\"\"\n    Complete Podcast Production Engine on 2x T4:\n    Dispatches asynchronous job so requests return in < 100ms, completely\n    bypassing Cloudflare Tunnel's 100-second idle timeout.\n    \"\"\"\n    try:\n        if llm_pipeline is None:\n            raise HTTPException(status_code=503, detail=\"LLM engine is initializing.\")\n\n        # Cleanup expired jobs older than 1 hour\n        now = time.time()\n        expired = [k for k, v in PODCAST_JOBS.items() if now - v.get(\"created_at\", now) > 3600]\n        for k in expired:\n            PODCAST_JOBS.pop(k, None)\n\n        job_id = f\"pod_{int(time.time())}_{os.urandom(3).hex()}\"\n        PODCAST_JOBS[job_id] = {\n            \"job_id\": job_id,\n            \"status\": \"processing\",\n            \"progress\": 0.05,\n            \"message\": \"Queued podcast generation on 2x T4 GPUs...\",\n            \"created_at\": now,\n        }\n\n        asyncio.create_task(_run_podcast_pipeline(job_id, req))\n\n        if sync:\n            while PODCAST_JOBS[job_id][\"status\"] == \"processing\":\n                await asyncio.sleep(1.0)\n            job = PODCAST_JOBS[job_id]\n            if job[\"status\"] == \"completed\":\n                return job[\"result\"]\n            raise HTTPException(status_code=500, detail=job.get(\"error\", \"Podcast failed\"))\n\n        return {\n            \"job_id\": job_id,\n            \"status\": \"processing\",\n            \"progress\": 0.05,\n            \"message\": \"Generating spoken script on 2x T4 GPUs...\",\n        }\n    except HTTPException:\n        raise\n    except Exception as e:\n        print(f\"Error in /api/podcast handler: {e}\")\n        import traceback\n        traceback.print_exc()\n        raise HTTPException(status_code=500, detail=f\"Podcast dispatch error: {str(e)}\")\n\n\n@app.get(\"/api/podcast/status/{job_id}\")\nasync def get_podcast_status(job_id: str):\n    job = PODCAST_JOBS.get(job_id)\n    if not job:\n        raise HTTPException(status_code=404, detail=\"Podcast job not found.\")\n    return job\n\n\n# ==============================================================================\n# Endpoint 6: Speech-to-Text Transcription\n# ==============================================================================\n@app.post(\"/v1/audio/transcriptions\")\nasync def audio_transcription(file: UploadFile = File(...)):\n    if whisper_model is None:\n        raise HTTPException(status_code=503, detail=\"Whisper STT model not loaded.\")\n\n    with tempfile.NamedTemporaryFile(\"wb\", suffix=f\"_{file.filename}\") as tmp:\n        content = await file.read()\n        tmp.write(content)\n        tmp_path = tmp.name\n\n    try:\n        segments, info = whisper_model.transcribe(tmp_path, beam_size=5)\n        text_parts = [segment.text for segment in segments]\n        full_text = \" \".join(text_parts).strip()\n        return {\"text\": full_text}\n    finally:\n        if os.path.exists(tmp_path):\n            os.remove(tmp_path)\n\n\n# ==============================================================================\n# Main Server Runner with Tunnel Support (Cloudflare Tunnel)\n# ==============================================================================\ndef start_tunnel_and_server(port: int = 8000):\n    print(\"\\n\" + \"=\" * 60)\n    print(\"\ud83d\ude80 Launching Cloudflare Tunnel for 2x T4 GPU Backend...\")\n    print(\"=\" * 60)\n\n    if not shutil.which(\"cloudflared\"):\n        print(\"Installing cloudflared binary...\")\n        subprocess.run(\n            [\"wget\", \"-q\", \"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\", \"-O\", \"cloudflared\"],\n            check=True\n        )\n        subprocess.run([\"chmod\", \"+x\", \"cloudflared\"], check=True)\n        cf_bin = \"./cloudflared\"\n    else:\n        cf_bin = \"cloudflared\"\n\n    tunnel_proc = subprocess.Popen(\n        [cf_bin, \"tunnel\", \"--url\", f\"http://localhost:{port}\"],\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        text=True,\n    )\n\n    tunnel_url = None\n    for _ in range(40):\n        time.sleep(0.5)\n        line = tunnel_proc.stderr.readline()\n        if \"trycloudflare.com\" in line:\n            match = re.search(r\"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com\", line)\n            if match:\n                tunnel_url = match.group(0)\n                break\n\n    if tunnel_url:\n        print(\"\\n\" + \"*\" * 60)\n        print(\"\ud83c\udf89 YOUR 2x T4 STUDYBOOK BACKEND IS LIVE!\")\n        print(\"\ud83d\udc49 Copy this Base URL into StudyBook Settings:\")\n        print(f\"   {tunnel_url}\")\n        print(\"\ud83d\udc49 Copy this API Key into StudyBook Settings (or KAGGLE_API_KEY):\")\n        print(f\"   {API_KEY}\")\n        print(\"*\" * 60 + \"\\n\")\n    else:\n        print(\"Could not auto-detect Cloudflare tunnel URL, check logs below.\")\n\n    import threading\n\n    config = uvicorn.Config(app, host=\"0.0.0.0\", port=port, log_level=\"info\")\n    server = uvicorn.Server(config)\n\n    # Detect if running inside a Jupyter / Kaggle notebook (IPython has an active event loop)\n    try:\n        loop = asyncio.get_running_loop()\n    except RuntimeError:\n        loop = None\n\n    if loop is not None and loop.is_running():\n        print(\"Detected running event loop (Jupyter/Kaggle notebook environment).\")\n        print(\"Launching Uvicorn server in a dedicated worker thread...\")\n        server_thread = threading.Thread(\n            target=lambda: asyncio.run(server.serve()),\n            daemon=True,\n        )\n        server_thread.start()\n        try:\n            while server_thread.is_alive():\n                time.sleep(1)\n        except KeyboardInterrupt:\n            print(\"\\nShutting down Kaggle backend server...\")\n    else:\n        server.run()\n\n\nif __name__ == \"__main__\":\n    start_tunnel_and_server()\n")

import importlib
import kaggle_backend_server
importlib.reload(kaggle_backend_server)
kaggle_backend_server.start_tunnel_and_server(port=8000)
